# Hessian K=2 Reduction Test

Verify that the general K-class Hessian implementation, when forced to K=2, matches the classical binary logistic regression formula:

$$S_N^{-1} = S_0^{-1} + \sum_{n=1}^N s_n(1-s_n) \phi_n \phi_n^T$$

where $s_n = \sigma(w^T \phi_n)$ is the sigmoid probability.

**Reference:** TODO.md Section 1b, item 1: "K=2 reduction matches the binary formula (notes §7.4)"

## Setup

In [1]:
import sys
from pathlib import Path

# Add parent directory to path
root_dir = Path().resolve().parent
sys.path.insert(0, str(root_dir))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.bayesian import FeatureClassifier, LastLayerLaplace

torch.manual_seed(42)
np.random.seed(42)

print("✓ Imports successful")

✓ Imports successful


## Generate Small Binary Dataset

In [2]:
# Small synthetic binary dataset
N = 20  # number of samples
D = 3   # feature dimension (without bias)
K = 2   # binary classification

# Generate random features
X = torch.randn(N, D)

# Generate binary labels
y = torch.randint(0, K, (N,))

print(f"Dataset: {N} samples, {D} features, {K} classes")
print(f"Class distribution: {(y == 0).sum()} vs {(y == 1).sum()}")

Dataset: 20 samples, 3 features, 2 classes
Class distribution: 6 vs 14


## Train Simple Binary Classifier

In [3]:
# Simple model: no hidden layers, direct linear classification
# This makes g(x) = x (identity feature extractor)
model = FeatureClassifier(in_dim=D, hidden_dims=[], n_classes=K)

# Train to MAP
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.01)
for epoch in range(100):
    optimizer.zero_grad()
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    optimizer.step()

# Evaluate
with torch.no_grad():
    preds = model(X).argmax(dim=1)
    acc = (preds == y).float().mean()
    
print(f"MAP training accuracy: {acc:.3f}")
print(f"Final loss: {loss.item():.4f}")

MAP training accuracy: 0.700
Final loss: 0.5790


## Compute Hessian via General K-class Formula

In [4]:
# Use the general implementation from bayesian.py
tau_prior = 0.01 * N  # weight_decay * N
laplace_general = LastLayerLaplace.fit(model, X, y, tau_prior=tau_prior)

H_general = laplace_general.cov  # Actually this is H^-1 (covariance)
H_general_inv = torch.linalg.inv(H_general)  # Get H itself

print(f"General K={K} Hessian shape: {H_general_inv.shape}")
print(f"Expected shape: {K * (D + 1)} x {K * (D + 1)} = {K * (D + 1)}^2")

General K=2 Hessian shape: torch.Size([8, 8])
Expected shape: 8 x 8 = 8^2


## Compute Hessian via Binary Logistic Formula

For binary logistic regression, the Hessian is:
$$H = \sum_{n=1}^N s_n(1-s_n) \phi_n \phi_n^T + \tau I$$

where $s_n$ are the sigmoid probabilities.

In [5]:
# Extract features with bias augmentation
with torch.no_grad():
    phi = model.features(X)  # Should be identity since no hidden layers
    # Add bias term
    ones = torch.ones(N, 1)
    phi_aug = torch.cat([phi, ones], dim=1)  # (N, D+1)
    
    # Get softmax probabilities
    logits = model(X)
    probs = F.softmax(logits, dim=1)  # (N, K=2)
    
    # For binary case: s_n(1-s_n) = p_n0 * p_n1
    # This is the diagonal element of Lambda_n for both classes
    s = probs[:, 1]  # probability of class 1
    weights = s * (1 - s)  # (N,)
    
    # Compute binary Hessian: sum_n s_n(1-s_n) phi_n phi_n^T + tau*I
    Dp = D + 1
    H_binary = torch.zeros(Dp, Dp)
    for n in range(N):
        H_binary += weights[n] * torch.outer(phi_aug[n], phi_aug[n])
    H_binary += tau_prior * torch.eye(Dp)

print(f"Binary Hessian shape: {H_binary.shape}")
print(f"Weights range: [{weights.min():.4f}, {weights.max():.4f}]")

Binary Hessian shape: torch.Size([4, 4])
Weights range: [0.1326, 0.2499]


## Analysis: Relation Between K=2 General and Binary Formula

For K=2 softmax, the general Hessian is a block matrix of size $2(D+1) \times 2(D+1)$.
The binary formula gives a $(D+1) \times (D+1)$ matrix.

We need to understand the relationship:
- The general formula has **redundancy** for K=2 (softmax constraint: $p_0 + p_1 = 1$)
- The binary formula parameterizes with a single weight vector

Let's examine the structure of the general Hessian for K=2.

In [6]:
# Extract blocks from the general K=2 Hessian
# H_general_inv is (2Dp, 2Dp), organized as [[H_00, H_01], [H_10, H_11]]
Dp = D + 1
H_00 = H_general_inv[:Dp, :Dp]
H_01 = H_general_inv[:Dp, Dp:]
H_10 = H_general_inv[Dp:, :Dp]
H_11 = H_general_inv[Dp:, Dp:]

print("Block structure of general K=2 Hessian:")
print(f"H_00 (class 0): {H_00.shape}")
print(f"H_01 (cross):   {H_01.shape}")
print(f"H_10 (cross):   {H_10.shape}")
print(f"H_11 (class 1): {H_11.shape}")
print()
print("Check symmetry: H_01 == H_10^T:", torch.allclose(H_01, H_10.T, atol=1e-5))
print("Check diagonal blocks equal: H_00 == H_11:", torch.allclose(H_00, H_11, atol=1e-5))

Block structure of general K=2 Hessian:
H_00 (class 0): torch.Size([4, 4])
H_01 (cross):   torch.Size([4, 4])
H_10 (cross):   torch.Size([4, 4])
H_11 (class 1): torch.Size([4, 4])

Check symmetry: H_01 == H_10^T: True
Check diagonal blocks equal: H_00 == H_11: True


## Verification: Compare Diagonal Blocks with Binary Formula

For K=2, the Hessian has a special structure. The diagonal blocks H_00 and H_11 should each match the binary formula (up to a possible sign or scaling).

In [7]:
# Check if H_00 matches H_binary
diff_00 = torch.abs(H_00 - H_binary).max()
rel_diff_00 = diff_00 / H_binary.abs().max()

# Check if H_11 matches H_binary  
diff_11 = torch.abs(H_11 - H_binary).max()
rel_diff_11 = diff_11 / H_binary.abs().max()

print("Comparison with binary formula:")
print(f"  Max absolute diff (H_00 vs H_binary): {diff_00:.2e}")
print(f"  Max relative diff (H_00 vs H_binary): {rel_diff_00:.2e}")
print(f"  Max absolute diff (H_11 vs H_binary): {diff_11:.2e}")
print(f"  Max relative diff (H_11 vs H_binary): {rel_diff_11:.2e}")
print()
print(f"  H_00 ≈ H_binary: {torch.allclose(H_00, H_binary, rtol=1e-4, atol=1e-6)}")
print(f"  H_11 ≈ H_binary: {torch.allclose(H_11, H_binary, rtol=1e-4, atol=1e-6)}")

Comparison with binary formula:
  Max absolute diff (H_00 vs H_binary): 5.72e-06
  Max relative diff (H_00 vs H_binary): 1.33e-06
  Max absolute diff (H_11 vs H_binary): 4.77e-06
  Max relative diff (H_11 vs H_binary): 1.11e-06

  H_00 ≈ H_binary: True
  H_11 ≈ H_binary: True


## Check Off-Diagonal Block Structure

For K=2, Lambda_n = diag(p_n) - p_n p_n^T has a specific structure:
$$\Lambda_n = \begin{bmatrix} p_0(1-p_0) & -p_0 p_1 \\ -p_0 p_1 & p_1(1-p_1) \end{bmatrix} = p_0 p_1 \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}$$

This means H_01 = H_10^T = -H_00.

In [8]:
# Check if off-diagonal blocks have the expected relationship
# Remove prior term for this check (prior only affects diagonal)
H_00_no_prior = H_00 - tau_prior * torch.eye(Dp)
H_01_no_prior = H_01

diff_off = torch.abs(H_01_no_prior + H_00_no_prior).max()
print(f"Check H_01 ≈ -H_00 (without prior): max diff = {diff_off:.2e}")
print(f"  Match: {torch.allclose(H_01_no_prior, -H_00_no_prior, rtol=1e-4, atol=1e-6)}")

Check H_01 ≈ -H_00 (without prior): max diff = 7.15e-07
  Match: True


## Summary and Conclusion

This test verifies that the general K-class Hessian implementation correctly reduces to the classical binary logistic regression formula when K=2.

**Expected results:**
1. The diagonal blocks H_00 and H_11 should both match the binary Hessian $\sum_n s_n(1-s_n) \phi_n \phi_n^T + \tau I$
2. The off-diagonal blocks should satisfy H_01 = H_10^T = -H_00 (without the prior term)
3. This confirms the implementation is correct for the binary case

**Interpretation:**
- For K=2, the softmax parameterization is redundant (only K-1=1 degrees of freedom needed)
- The general Hessian encodes this redundancy in its block structure
- Each diagonal block recovers the classical binary formula
- The symmetry confirms the implementation is mathematically consistent

In [9]:
# Final assertion for automated testing
print("\n" + "="*60)
print("FINAL VERIFICATION")
print("="*60)

tolerance_passed = torch.allclose(H_00, H_binary, rtol=1e-4, atol=1e-6)
structure_passed = torch.allclose(H_01_no_prior, -H_00_no_prior, rtol=1e-4, atol=1e-6)

if tolerance_passed and structure_passed:
    print("✓ TEST PASSED: K=2 general Hessian matches binary formula")
    print("  - Diagonal blocks match classical binary Hessian")
    print("  - Off-diagonal structure is correct (H_01 = -H_00)")
    print("\nThe general K-class implementation is VERIFIED for K=2.")
else:
    print("✗ TEST FAILED")
    if not tolerance_passed:
        print("  - Diagonal blocks do NOT match binary formula")
    if not structure_passed:
        print("  - Off-diagonal structure is incorrect")
    print("\nReview the Hessian implementation in src/bayesian.py")


FINAL VERIFICATION
✓ TEST PASSED: K=2 general Hessian matches binary formula
  - Diagonal blocks match classical binary Hessian
  - Off-diagonal structure is correct (H_01 = -H_00)

The general K-class implementation is VERIFIED for K=2.
